# Day 2 — EDA 질문 뱅크 (노트북 버전)

> **EDA의 핵심은 좋은 질문 — AI는 코드를 짠다, 질문은 분석가의 일.**
> 같은 데이터라도 어떤 질문을 던지느냐에 따라 인사이트가 갈린다.

## 이 노트북

`prompts/eda_question_bank.md`의 50개 질문을 Colab 셀 형태로 펼친 버전입니다.
- 본인 관심사 5개 골라 Claude Code에 그대로 던지면 끝 — **분석 코드는 Claude가 짭니다**
- Claude가 답한 코드를 빈 셀에 붙여 ▶︎ 실행하고 결과 확인
- 모든 50개 답을 한 번에 보고 싶으면 → `b2g_day2_eda_solutions.ipynb`

## 사전 준비
좌측 폴더 아이콘 → 다음 CSV 업로드 (강의 레포 `data/` 폴더에서 받기 가능):
- `기간별_일평균_대기환경_정보_2024년.csv`, `_2025년.csv` (CP949)
- `seoul_bike_daily_2024.csv` (CP949)
- `seoul_subway_daily_total_2024.csv` (CP949)
- `OBS_ASOS_DD_20260510160758.csv` (CP949)
- `time_series_KR_20240101-0000_20260510-1610.csv` (UTF-8)


In [ ]:
# 한글 폰트 (Colab)
!apt -qq install fonts-nanum > /dev/null
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

import pandas as pd, numpy as np
print('OK')

## 데이터 로드 (한 번에)

In [ ]:
# 미세먼지 (2024 + 2025)
a = pd.read_csv('기간별_일평균_대기환경_정보_2024년.csv', encoding='cp949')
b = pd.read_csv('기간별_일평균_대기환경_정보_2025년.csv', encoding='cp949')
a = a.rename(columns={'측정일시':'date','측정소명':'district','미세먼지(㎍/㎥)':'pm10','초미세먼지(㎍/㎥)':'pm25'})[['date','district','pm10','pm25']]
b = b.rename(columns={'측정일시':'date','측정소명':'district','미세먼지농도(㎍/㎥)':'pm10','초미세먼지농도(㎍/㎥)':'pm25'})[['date','district','pm10','pm25']]
air = pd.concat([a, b], ignore_index=True)
air['date'] = pd.to_datetime(air['date'].astype(str), format='%Y%m%d')
air['year'] = air['date'].dt.year; air['month'] = air['date'].dt.month
air['weekday'] = air['date'].dt.weekday
air['season'] = air['month'].map(lambda m: '황사(3-5월)' if 3<=m<=5 else '비황사')

# 따릉이 / 지하철 (2024년)
bike = pd.read_csv('seoul_bike_daily_2024.csv', encoding='cp949').rename(columns={'대여일자':'date','대여건수':'rides'})
bike['date'] = pd.to_datetime(bike['date'])
sub = pd.read_csv('seoul_subway_daily_total_2024.csv', encoding='cp949').rename(columns={'수송일자':'date','총승하차인원':'passengers'})[['date','passengers']]
sub['date'] = pd.to_datetime(sub['date'])

# 날씨 (ASOS 2024+2025)
weather = pd.read_csv('OBS_ASOS_DD_20260510160758.csv', encoding='cp949').rename(columns={
    '일시':'date','평균기온(°C)':'temp_avg','최저기온(°C)':'temp_min','최고기온(°C)':'temp_max',
    '일강수량(mm)':'rain','평균 풍속(m/s)':'wind_avg','최다풍향(16방위)':'wind_dir','평균 상대습도(%)':'humidity_avg',
})[['date','temp_avg','temp_min','temp_max','rain','wind_avg','wind_dir','humidity_avg']]
weather['date'] = pd.to_datetime(weather['date'])

# 검색량 (구글 트렌드 월별)
buzz = pd.read_csv('time_series_KR_20240101-0000_20260510-1610.csv')
buzz.columns = ['date','buzz']; buzz['date'] = pd.to_datetime(buzz['date'])

# 일평균 PM2.5
daily_pm = air.groupby('date')['pm25'].mean().reset_index()
daily_pm['grade'] = pd.cut(daily_pm['pm25'], bins=[-1,15,35,75,1e9], labels=['좋음','보통','나쁨','매우나쁨'])

print('air:', air.shape, '· bike:', bike.shape, '· sub:', sub.shape, '· weather:', weather.shape, '· buzz:', buzz.shape)

---

# A. 미세먼지 단독 (10개)


**A1. 두 해 통합 shape — 행 수 / 컬럼 수 / 측정소 수 / 날짜 범위**

**A2. 컬럼별 결측 비율 / 50% 이상 결측 측정소가 있나?**

**A3. PM2.5·PM10 분포 (히스토그램 + skew)**

**A4. 자치구별 PM2.5 연평균 Top 5 / Bottom 5 / 격차**

**A5. 2024 → 2025 자치구별 PM2.5 변화율 — 가장 좋아진 곳·나빠진 곳**

**A6. 월별 PM2.5 평균/표준편차 — 시즌성 패턴**

**A7. 황사 시즌 vs 비황사 시즌 PM10 분포 (박스플롯)**

**A8. 요일별 PM2.5 평균 — 평일 vs 주말 t-test**

**A9. PM10·PM2.5 상관계수 + 산점도**

**A10. 일별 spike Top 10 — 진짜 이상치인가, 황사인가?**

---

# B. 미세먼지 + 따릉이 (5개) — 2024년 매칭


**B11. PM2.5 등급별 따릉이 일평균 대여 건수**

**B12. 미세먼지 등급 1단계 악화 시 따릉이 대여가 몇 % 감소?**

**B13. 황사 vs 비황사 시즌 따릉이 이용 + t-test**

**B14. 미세먼지 영향이 더 큰 시즌은? (계절별)**

**B15. "PM2.5 30 이하 + 맑음" 조합에서 대여가 폭발하나?** (조건 결합)

---

# C. 미세먼지 + 지하철 (5개) — 2024년 매칭


**C16. PM2.5 등급별 지하철 일별 총 승하차 — 회피 효과 있나?**

**C17. 따릉이=민감 / 지하철=둔감 — 대체재 가설 검증**

**C18. 황사 spike 일자에 따릉이↓ + 지하철↑ 동시 일어나나?**

**C19. 출근시간 7~9시 PM2.5와 지하철 승차 관계** ⛔

**답 안 됨**: AQ 데이터는 일별이라 시간대 분석 불가. 시간대 분석을 원하면 에어코리아 시간별 OpenAPI를 별도로 받아야 함.

**C20. 평일 vs 주말 — 두 데이터 다 평일이 더 높나? (커뮤팅)**

---

# D. 미세먼지 + 날씨 (8개) — 2024 + 2025


**D21. 풍속 → PM2.5 음의 상관 / 바람이 셀수록 농도 떨어지나?**

**D22. 강수일 vs 비강수일 PM2.5 분포 (비가 씻어내나?)**

**D23. 풍향(서·북서 vs 동) 별 PM10 — 황사 유입 검증**

**D24. 기온·습도와 PM 상관 — 겨울 난방 효과?**

**D25. "맑고 풍속 약함 + 한파" 조합에서 PM 폭증?** (정체)

**D26. 황사 시즌 풍향 분포 — 작년과 다른가?**

**D27. 일교차 큰 날 PM 더 높나? (대기 정체)**

**D28. "다음 날 PM 예측" — 오늘 날씨 변수로 회귀 R²**

---

# E. 미세먼지 + 언급량 (4개) — 월 단위


**E29. "미세먼지" 검색량과 PM2.5 일별 상관 (월 단위로 집계)**

**E30. 사람들은 PM 몇 ㎍/㎥부터 검색하기 시작하나? (임계값)**

**E31. "마스크"·"공기청정기" 검색 급등 vs PM 실측 — 어느 게 며칠 빠른가?**

**별도 키워드 buzz 데이터 필요** — 현재는 "미세먼지" 단일 키워드만 받음. 네이버 데이터랩에서 "마스크", "공기청정기" 키워드 추가로 받아 동일 흐름으로 분석 가능.

**E32. 같은 PM 농도에도 시즌·요일에 따라 검색 강도 다른가?**

---

# F. 쇼핑 적재본 (8개) — Day 4

> 쇼핑 적재본(다이소·올리브영·컬리·네이버쇼핑)은 4일차 결합용으로 강의 기간 누적.
> Day 2 시점에는 데이터 미보유. Day 4 노트북에서 다룸.


---

# G. 답 안 되는 질문 (5개) — 한계 인식

| 질문 | 왜 답 못 함 |
|---|---|
| 출근시간(7~9시) vs 퇴근시간 PM | AQ 일별 — 시간대 분석 불가. 에어코리아 시간별 OpenAPI 별도 |
| 코로나 봉쇄 전후(2020~2022) PM 변화 | 배포본은 2024~2025만. 과거 자료 별도 다운로드 |
| 자치구별 사망률·호흡기질환 상관 | 보건 데이터 별도 — 강의 범위 외 |
| 중국 황사 발원지 풍향 | 위성·재분석 데이터 필요 — 강의 범위 외 |
| 학생 본인 회사 매출과 PM 상관 | 사내 데이터 — 미션 보너스 영역 |


---

# H. 워밍업 (5개)


**H46. 내가 사는 자치구의 2025 PM2.5 평균은?** — 본인 자치구로 변경

**H47. 어제 vs 지난주 같은 요일 비교** — 짧은 기간 변동 체감 (데이터 마지막 날 기준)

**H48. 가장 깨끗했던 하루 + 그날 어떤 날씨였는지**

**H49. 황사 경보 수준이었던 날 따릉이 대여가 정말 줄었나?**

**H50. 한 달 뒤 PM2.5 예측 — 가진 변수로 어디까지 가능한가?**

**Day 3·4 예고**: 이 노트북의 D28 회귀 모델을 확장해서 한 달 뒤 PM 예측. 가진 변수로 R² ~0.3 정도가 한계 — 더 나아가려면 위성·외기 데이터 결합 필요. Day 4 의사결정 리포트와 연결.

---

## CLAUDE.md 누적

오늘 발견한 규칙을 자연어 한 마디로 CLAUDE.md에 추가:

```
오늘 EDA에서 발견한 규칙을 CLAUDE.md에 추가해줘.
- 측정일시는 항상 datetime 파싱 (format='%Y%m%d')
- 결측 50% 이상 측정소 제외
- 이상치 IQR 3배 (황사 spike 인정), 황사 시즌 = 3~5월
- 결합 데이터 날짜 컬럼은 'date'로 통일
- 미세먼지는 CP949, buzz는 UTF-8
- y축 단위 항상 명시 (PM은 ㎍/㎥, 따릉이는 건, 지하철은 명)
```
